<a href="https://colab.research.google.com/github/MuhammadAli055/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

!git clone https://github.com/MuhammadAli055/flyrank-ml-internship.git

os.chdir('/content/flyrank-ml-internship')

print(os.listdir())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 23), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.84 MiB | 8.82 MiB/s, done.
Resolving deltas: 100% (23/23), done.
['LICENSE', 'notebooks', 'skills', '.github', 'CLAUDE.md', 'data', '.git', 'submission', 'AGENTS.md', 'docs', 'requirements.txt', 'outputs', 'README.md', 'work', 'DATA_USE.md', 'GUIDE.md', '.gitignore', 'SETUP.md', 'scripts']


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## Section 1: My Lane as an ML Task

**Lane chosen:** Lane 2 — Refresh / Content Opportunity Scoring

**Task type:** Binary Classification + Ranking/Scoring

This is a binary classification problem at its core. For each content page,
the model answers one question: is this page likely to need review or not?
The output is a probability score between 0 and 1. Pages are then ranked
from highest to lowest score to produce a prioritized review queue.

It is not just classification because the final output is a RANKED LIST,
not just a yes/no label. The probability score is what creates the ranking —
higher score means review this page first.

It is not clustering, because we are not grouping pages into types.
It is not pure ranking, because we start from a classification signal
and turn it into a ranked queue.

In plain words: for every content page, produce a score from 0 to 100
that tells a content reviewer how urgently that page needs attention,
and sort them from most urgent to least urgent.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Section 2: Target or Proxy

**My target (proxy label):**
is_declining_label = (trend_direction == "down")

This means: a page is labeled positive (1) if its traffic trend direction
is currently "down", and negative (0) otherwise.

**Why it is called a proxy:**
This is not a perfect label. Ideally, I would define a future-looking label like:
"did this page lose more than X% of impressions over the next 30 days?"
That would require knowing the future, which I don't have yet.

The current proxy uses the trend direction calculated from the existing data window.
It is a beginner proxy — a stand-in for the real thing. It is honest and useful
for building and testing the pipeline now, and I will aim to strengthen it
with future-window labels in later weeks when I use the full warehouse data.

**What the label is NOT:**
- It is not a guarantee that this page needs a rewrite
- It is not proof that refreshing the page will fix it
- It is not a product decision — it is an observed measurement from real data

**Possible stronger label for later weeks:**
features from the prior 90 days → did the page decline over the next 30 days?
That future-window design avoids using the target window as a feature,
making it a much cleaner and more honest label.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Section 3: Success Metric

**Primary metric: Precision@K (specifically Precision@50)**

Precision@50 asks: of the top 50 pages the model says to review first,
how many of them actually turned out to be genuine candidates?

**Why Precision@K and not just accuracy?**
A content team can only review a limited number of pages per week —
maybe 20, maybe 50. They will not go through hundreds of pages.
So what matters is: are the pages at the TOP of the list the right ones?

Generic accuracy would not capture this. A model could be 90% accurate
overall by just predicting "no action needed" for everything —
but that would be completely useless for a reviewer.

Precision@K directly measures what the reviewer experiences:
are the first pages I open worth my time?

**Secondary metrics I will also track:**
- ROC AUC: measures overall ranking quality across all thresholds
- Average Precision: measures quality across the whole ranked list
- Recall: measures how many true candidates we actually caught

**Baseline to beat (from notebook 01 results):**
- Baseline rule Precision@50 = 0.240 (~12 out of 50 correct)
- Random forest Precision@50 = 0.740 (~37 out of 50 correct)

My goal is to match or beat the random forest result with a well-validated,
explainable model that also produces clear reason codes.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create the target column
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Show unit of analysis: one row = one content page
print(f"Total pages in dataset: {len(df)}")
print(f"\nOne row = one content page. Here are the key columns:\n")

# Select the most important columns to display
key_columns = [
    'content_id',
    'impressions_90d',
    'sessions_90d',
    'avg_position',
    'ctr',
    'content_age_days',
    'trend_direction',
    'is_declining_label'
]

# Show first 5 rows
print(df[key_columns].head())

# Show class balance
print(f"\nTarget label breakdown:")
print(df['is_declining_label'].value_counts())
print(f"\nDecline rate: {df['is_declining_label'].mean()*100:.1f}% of pages are declining")

Total pages in dataset: 30000

One row = one content page. Here are the key columns:

             content_id  impressions_90d  sessions_90d  avg_position   ctr  \
0  content_304f48230142             3803            17          10.6  0.76   
1  content_a1fb4e703a9e            15320             9          20.3  0.05   
2  content_9aa793d4d895            12581            11          36.5  0.09   
3  content_331d6c4de07b            11751            78           6.2  0.49   
4  content_d99b7a2d90ca            19140           145          44.0  0.13   

   content_age_days trend_direction  is_declining_label  
0               187            down                   1  
1               445            down                   1  
2               141            down                   1  
3               463          stable                   0  
4               263            down                   1  

Target label breakdown:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Decl

## What The Dataframe Shows

**One row = one content page.**

Each row represents a single content page and contains:
- content_id: a unique identifier for the page (anonymized)
- impressions_90d: how many times the page appeared in search results in the last 90 days
- sessions_90d: how many times people actually visited the page in the last 90 days
- avg_position: the average position this page ranks at in Google search results
- ctr: click-through rate — what fraction of people who saw it actually clicked
- content_age_days: how old the content is in days
- trend_direction: whether traffic is going up, down, or staying flat
- is_declining_label: our target — 1 means declining, 0 means not declining

This is the grain of the whole project. Every score, every prediction,
and every ranked recommendation will be at this level — one row per page.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Section 5: Why ML Beats a Fixed Rule Here

**The fixed rule approach:**
A human expert writes conditions like:
- IF content_age_days > 180 AND impressions_90d > 500 → flag for review
- IF trend_direction == "down" AND impressions_90d > 100 → flag for review

This is the baseline score already built in the starter pipeline.
It is transparent and easy to explain — but it has a hard limit.

**The problem with fixed rules:**
A fixed rule can only check one or two conditions at a time.
Real decline patterns involve many signals interacting together:
a page might have decent impressions but terrible CTR AND be very old
AND have declining sessions AND have a weak engagement rate —
and none of those signals alone would trigger the rule,
but together they clearly point to a page that needs attention.

A fixed rule cannot learn these combinations. It treats every signal
independently with hand-tuned weights that someone guessed.

**What ML does differently:**
A machine learning model learns which combinations of signals —
across all columns at once — are actually associated with pages
that need review. It does not need a human to guess the weights.
It finds the patterns from the data itself.

**The proof — from notebook 01 results:**

| Method | Precision@50 | Pages correct out of top 50 |
|---|---|---|
| Hand-written baseline rule | 0.240 | ~12 out of 50 |
| Logistic Regression | 0.400 | ~20 out of 50 |
| Decision Tree | 0.540 | ~27 out of 50 |
| Random Forest | 0.740 | ~37 out of 50 |

The random forest found 3x more real candidates in the top 50
than the fixed rule did — on the same data. That gap is the value
of learning from signals instead of guessing rules by hand.

**But ML is not magic:**
The model still needs human judgment at the end. It surfaces candidates —
a reviewer decides what to actually do with each page.
The model improves WHERE reviewers look, not WHAT they decide.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Section 6: Self-Check

- [x] I named the ML task type: Binary Classification + Ranking/Scoring
- [x] I named the target/proxy: is_declining_label = (trend_direction == "down")
- [x] I explained why it is a proxy and what a stronger label would look like
- [x] I named the success metric: Precision@K (Precision@50)
- [x] I explained WHY Precision@K fits this problem better than accuracy
- [x] I showed the unit of analysis as a real dataframe: one row = one content page
- [x] I showed the target column in the dataframe
- [x] I explained why ML beats a fixed rule — with real numbers as proof
- [x] I tied the output to a real content action: ranked review queue for content team
- [x] I used careful language throughout: "associated with", "suggests", "observed"